In [1]:
import math
from typing import Any, Callable

import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.types import Tensor

import spark as sp

In [ ]:
def linear_scheduler(timesteps: int, start: float, stop: float) -> Tensor:
    """
    Linear noise schedule for the forward diffusion process.
    From DDPM (Ho et al, 2020): https://arxiv.org/pdf/2006.11239.
    """
    return torch.linspace(start, stop, timesteps)

def cosine_scheduler(timesteps: int, s: float = 0.008) -> Tensor:
    """
    Cosine noise schedule for the forward diffusion process.
    From Improved DDPM (Nichol, Dhariwal, 2021): https://arxiv.org/abs/2102.09672.
    """
    x = torch.linspace(0, timesteps, timesteps + 1) / timesteps
    x = torch.cos(0.5 * torch.pi * (x + s) / (1 + s)).pow(2)
    x = x / x[0]
    betas = 1 - (x[1:] / x[:-1])
    return torch.clip(betas, 1e-4, 0.9999)

In [ ]:
class RegistryTable:
    """
    Container with pre-computed quantities for the diffusion process.
    """
    def __init__(self, betas: Tensor) -> None:
        # define alphas
        alphas = 1.0 - betas
        alphas_cumprod = torch.cumprod(alphas, axis=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
        self.sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
        # calculations for diffusion q(x_t | x_{t-1}) and others
        self.sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
        self.sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
        # calculations for posterior q(x_{t-1} | x_t, x_0)
        self.posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)